In [2]:
import json

In [10]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI

In [3]:
file_path = r'C:\dev\lv-assignment1\dummybot.json'
with open(file_path, 'r') as file:
    data = json.load(file)

In [4]:
data

[{'id': 1,
  'question': 'What is the best time to visit Paris?',
  'expected_answer': 'The best time to visit Paris is from April to June and September to October when the weather is pleasant and crowds are smaller.',
  'bot_answer': 'April to June and September to October are ideal months due to good weather and fewer tourists.'},
 {'id': 2,
  'question': 'Do Indian citizens need a visa to travel to Thailand?',
  'expected_answer': 'Indian citizens currently can obtain a visa on arrival or may travel visa-free depending on updated government policies.',
  'bot_answer': 'No visa is required for Indian citizens to visit Thailand.'},
 {'id': 3,
  'question': 'What currency is used in Japan?',
  'expected_answer': 'The currency used in Japan is the Japanese Yen (JPY).',
  'bot_answer': 'Japan uses Yen.'},
 {'id': 4,
  'question': 'How many days are enough to explore Rome?',
  'expected_answer': '3 to 4 days are generally enough to explore Romeâ€™s main attractions like the Colosseum, Vat

LLM as a Judge

In [14]:
Instruction ="""
You are an expert Travel Assistant acting as a judge to evaluate another 
travel assistant's response.
You will be given:

A user question
{question}
The expected answer
{ground_truth}
The bot's answer
{bot_answer}

Your task is to evaluate the bot's answer in the following categories:
Correctness : Does the bot answer the question accurately?
Relevance : Is the response directly relevant to the question?
Completeness :Does the response fully address all aspects of the question?
Clarity :Is the response clear, well-structured, and easy to understand?

Rate each category on a scale of 1 to 5, where:

1 = Very Poor
2 = Poor
3 = Average
4 = Good
5 = Excellent

Finally, provide:
Individual scores for each category
A short justification (1 to 2 sentences per category)
An overall final rating (1 to5)
Be objective, unbiased, and consistent in your evaluation.

"""

In [11]:
llm=ChatOpenAI(model="gpt-4o-mini")

In [12]:
from pydantic import BaseModel, Field

class EvaluationResult(BaseModel):
    correctness: int = Field(..., description="Score for correctness (1-5)")
    relevance: int = Field(..., description="Score for relevance (1-5)")
    completeness: int = Field(..., description="Score for completeness (1-5)")
    clarity: int = Field(..., description="Score for clarity (1-5)")
    justification: str = Field(..., description="Justification for the scores")
    overall_rating: int = Field(..., description="Overall final rating (1-5)")

In [13]:
llm_response = llm.with_structured_output(EvaluationResult)

In [15]:
prompt = ChatPromptTemplate.from_template(Instruction)

In [45]:
import warnings
warnings.filterwarnings("ignore", category=UserWarning)

In [46]:
output=[]
for idx,i in enumerate(data):
    id=i['id']
    question=i['question']
    ground_truth=i['expected_answer']
    bot_answer=i['bot_answer']
    evaluation=chain.invoke({"question":question, 
    "ground_truth":ground_truth, "bot_answer":bot_answer})
    correctness_score=evaluation.correctness
    relevance_score=evaluation.relevance
    completeness_score=evaluation.completeness
    clarity_score=evaluation.clarity
    justification=evaluation.justification
    overall_rating=evaluation.overall_rating
    output.append({"id":id, "question":question,"ground_truth":ground_truth, 
    "bot_answer":bot_answer, "correctness_score":correctness_score,"relevance_score":relevance_score,
     "completeness_score":completeness_score, "clarity_score":clarity_score,
     "justification":justification, "overall_rating":overall_rating})
    if (idx+1) % 5==0:
        print(f"Processing of {idx+1} record completed")

Processing of 5 record completed
Processing of 10 record completed
Processing of 15 record completed
Processing of 20 record completed
Processing of 25 record completed


In [47]:
output

[{'id': 1,
  'question': 'What is the best time to visit Paris?',
  'ground_truth': 'The best time to visit Paris is from April to June and September to October when the weather is pleasant and crowds are smaller.',
  'bot_answer': 'April to June and September to October are ideal months due to good weather and fewer tourists.',
  'correctness_score': 5,
  'relevance_score': 5,
  'completeness_score': 4,
  'clarity_score': 5,
  'justification': "Correctness: The bot accurately identifies the best months to visit Paris as April to June and September to October, matching the expected answer. Relevance: The response is directly relevant to the question regarding the best time to visit Paris. Completeness: While the bot provides the ideal months, it could have briefly mentioned the reasons for fewer tourists or pleasant weather for additional context. Clarity: The bot's response is clear and well-structured, making it easy to understand.",
  'overall_rating': 4},
 {'id': 2,
  'question': '

In [48]:
import pandas as pd

In [49]:
evaluation_result=pd.DataFrame(output)

In [52]:
evaluation_result.to_csv("evaluation_result.csv",index=False)